# Compare controlled runs and check recommendations

Linden needs a defensible choice rather than five unrelated scores. The fixture holds data, split, scaling, solver and guard fixed while changing alpha.

**Local:** [set up once](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/docs/setup.md), select the module `.venv` kernel, then run cells in order. **Colab:** CPU, runtime version **2026.07** (Python 3.12). If setup asks for a restart after installing dependencies, use **Restart session**, then run again. No API key is needed. Setup prints a new temporary output folder; save your work before ending the session.

Use this one notebook throughout the sprint: predict → run the guided cells → add your answers in **Your turn** → finish the checkpoint. Campus notes explain the same work; you do not repeat it in a second notebook.

<details><summary>Campus notes for this sprint</summary>

- [Why ad hoc tracking breaks down at scale](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-2/01-why-ad-hoc-tracking-breaks-down-at-scale/lesson.md)
- [Controls, confounders, and decision metrics](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-2/02-controls-confounders-and-decision-metrics/lesson.md)
- [Building the five-run experiment matrix](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-2/03-building-the-five-run-experiment-matrix/lesson.md)
- [Revisiting MLflow in a pipeline context](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-2/04-revisiting-mlflow-in-a-pipeline-context/lesson.md)
- [Comparing multiple runs systematically](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-2/05-comparing-multiple-runs-systematically/lesson.md)
- [Autologging and reducing manual tracking overhead](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-2/06-autologging-and-reducing-manual-tracking-overhead/lesson.md)
- [Validating an assistant-generated experiment summary](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-2/07-validating-an-assistant-generated-experiment-summary/lesson.md)
- [Experiment tracking deliverable](https://github.com/richhiey/ml-llm-ops/blob/main/module-a/lessons/sprint-2/08-experiment-tracking-deliverable/lesson.md)

</details>

In [1]:
from pathlib import Path
import json, tempfile, sys, subprocess
PIN = {'package': 'campus-mlops', 'version': '0.1.1', 'code_revision': 'sha256:a7a3061850f1f1eaa158754e719262b26a000af73019ada402be743c6f66e6c2', 'repository': 'https://github.com/richhiey/ml-llm-ops.git', 'source_commit': 'f1ae85914f49a4f1c4bdd6a0ca62c4c4076158ea', 'package_path': 'module-a'}
assert sys.version_info[:2] == (3, 12), 'Use Python 3.12; see the linked setup guide.'
if 'google.colab' in sys.modules:
    # Start before importing numerical packages: installed files cannot replace live modules.
    expected = {'numpy':'2.3.3', 'pandas':'2.3.3', 'sklearn':'1.7.2', 'mlflow':'3.5.1'}
    stale = [name for name, version in expected.items()
             if name in sys.modules and getattr(sys.modules[name], '__version__', None) != version]
    checkout = Path('/content') / ('ml-llm-ops-' + PIN['source_commit'][:12])
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--no-checkout', PIN['repository'], str(checkout)], check=True)
    subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', PIN['source_commit']], check=True)
    actual = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'], text=True).strip()
    assert actual == PIN['source_commit']
    package = checkout / PIN['package_path']
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(package/'requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest==8.4.2'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(package)], check=True)
    if stale:
        raise RuntimeError('Pinned dependencies are now installed. Choose Runtime > Restart session, '
                           'then Run all again. Keep this runtime: do NOT disconnect/delete it. '
                           'Loaded packages needing refresh: ' + ', '.join(stale))
    # A running kernel does not reload newly installed editable .pth files.
    sys.path.insert(0, str(package / 'src'))
import campus_mlops
from campus_mlops import *
from campus_mlops.lifecycle import code_revision, MODEL_NAME
PACKAGE=Path(campus_mlops.__file__).resolve().parents[2]
sys.path.insert(0,str(PACKAGE))
from learning.workspace import make_project, project_python, project_tests, course_cli, prepare_case
assert campus_mlops.__version__ == PIN['version'] == '0.1.1'
assert code_revision() == PIN['code_revision'], 'Helper changed: restore the pinned reference source.'
state = Path(tempfile.mkdtemp(prefix='campus-module-a-'))
print('Helper', campus_mlops.__version__, 'source commit:', PIN['source_commit'], 'state:', state)


Helper 0.1.1 source commit: f1ae85914f49a4f1c4bdd6a0ca62c4c4076158ea state: /var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-x4mw3x4j


## Calculate the metric distinction

Illustrative errors A=[2,2,8] and B=[4,4,4] have the same average absolute error. Predict which has larger RMSE and why before calculating. These are teaching values, not experimental scores.

In [2]:
import numpy as np
for label, errors in [('A',[2,2,8]),('B',[4,4,4])]:
    errors=np.asarray(errors,dtype=float)
    print(label,'MAE',np.abs(errors).mean(),'RMSE',np.sqrt((errors**2).mean()))

A MAE 4.0 RMSE 4.898979485566356
B MAE 4.0 RMSE 4.0


## Plan before running

Record five positive `alpha` values, including `1` as the anchor. Hold the snapshot, split, feature order, scaling, solver and guard fixed. State the primary validation metric and its direction, then label one hypothetical changed-partition comparison as excluded. You may use the supplied five values; complete this plan before running the next cell. Keep the final holdout closed.

## Run the declared matrix
Autologging hooks model fitting. We disable automatic model logging inside the helper so the explicit named model remains unambiguous. The helper still logs the primary held-out metric itself.

The public `run_matrix` fixture supplies five alpha values; your independent matrix may choose another declared set. Keep the returned IDs: the store can contain other successful executions. Parameters and metrics are not the fitted artifact; the model URI and recovery files carry the next links.

In [3]:
ws=Workspace(state/'tracking')
records=run_matrix(ws,autolog=True)
selected_ids=[record['run_id'] for record in records]
(ws.path/'matrix-run-ids.json').write_text(json.dumps(selected_ids,indent=2))
all_rows=export_runs(ws)
rows=all_rows[all_rows.run_id.isin(selected_ids)].copy()
rows.to_csv(ws.path/'runs.csv',index=False)
display(rows[['run_id','alpha','range_guard','evaluation_partition','rmse_kwh','mae_kwh']].sort_values('rmse_kwh'))
assert len(rows)==5

2026/10/19 17:38:15 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/19 17:38:15 INFO mlflow.store.db.utils: Updating database tables


2026-10-19 17:38:15 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.


2026-10-19 17:38:15 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


2026-10-19 17:38:15 INFO  [alembic.runtime.migration] Running upgrade  -> 451aebb31d03, add metric step


...

2026/10/19 17:38:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/19 17:38:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/19 17:38:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/10/19 17:38:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


,run_id,alpha,range_guard,evaluation_partition,rmse_kwh,mae_kwh
4,32d4a31f0635446eb714e32f58a7eaca,0.01,True,validation,1.434282,1.201891
3,647bae2c99ae4297be25d1da3b33ff8c,0.1,True,validation,1.434794,1.203859
2,f615548053b7426c952240c66ce88da3,1.0,True,validation,1.443004,1.223427
1,bec6aa8845254f619384bf18276a95e5,10.0,True,validation,1.766063,1.437359
0,6630883d65784815875f72c0c54aa127,100.0,True,validation,6.715349,5.680055


## Inspect raw MLflow evidence
Auto-generated training metrics are useful diagnostics, not the declared decision metric. Inspect a raw run and artifact list instead of trusting a summary.

In [4]:
best=rows.sort_values('rmse_kwh').iloc[0]
raw=ws.client.get_run(best.run_id)
print('alpha:',best.alpha,'validation RMSE:',best.rmse_kwh)
print('Autolog parameter:',raw.data.params['ridge__alpha'])
print('Training metric:',raw.data.metrics['training_mean_squared_error'])
print([a.path for a in ws.client.list_artifacts(best.run_id,'reproduction')])
assert best.rmse_kwh==rows.rmse_kwh.min()

alpha: 0.01 validation RMSE: 1.4342817684933908
Autolog parameter: 0.01
Training metric: 2.0040508496474954
['reproduction/config.json', 'reproduction/environment.txt', 'reproduction/manifest.json', 'reproduction/source.tar.gz']


## Compare logging with the model fixed

Reuse the matrix run with alpha=1 as the autologging example. Run just one matching configuration with logging off. This pair investigates logging while keeping the model and validation rows fixed. Predict whether validation RMSE should change, then compare the recorded key sets.

In [5]:
off=ws.run(Config(alpha=1,range_guard=True,autolog=False),name='logging-off')
on=next(record for record in records if float(record['config']['alpha'])==1.0)
off_raw=ws.client.get_run(off['run_id']).data
on_raw=ws.client.get_run(on['run_id']).data
print('Added parameters:',sorted(set(on_raw.params)-set(off_raw.params)))
print('Added metrics:',sorted(set(on_raw.metrics)-set(off_raw.metrics)))
assert 'ridge__alpha' in on_raw.params
assert abs(off['metrics']['rmse_kwh']-on['metrics']['rmse_kwh'])<=1e-6
assert len(rows)==5

/Users/richhiey/Desktop/projects/masterschool/lessons/mlops-llmops/shared/ml-llm-ops/module-a/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/19 17:38:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a v

Added parameters: ['memory', 'ridge', 'ridge__alpha', 'ridge__copy_X', 'ridge__fit_intercept', 'ridge__max_iter', 'ridge__positive', 'ridge__random_state', 'ridge__solver', 'ridge__tol', 'scale', 'scale__copy', 'scale__with_mean', 'scale__with_std', 'steps', 'transform_input', 'verbose']
Added metrics: ['training_mean_absolute_error', 'training_mean_squared_error', 'training_r2_score', 'training_root_mean_squared_error', 'training_score']


## Validate an assistant recommendation
Export `runs.csv` to your institution-approved assistant only if its data policy permits. Ask it to name run IDs, metric direction, controls and caveats. Paste the actual response into this notebook. The following intentionally wrong recommendation is a deterministic test case, not a claimed live assistant call.

Split each actual response into claims: winner/metric, controls and operational recommendations. Check every claim against a source field or name the missing evidence. The code below checks numeric selection only; it does not establish that you directed an assistant or reviewed its prose.

In [6]:
wrong=rows.sort_values('rmse_kwh').iloc[-1]
print(verify_summary(rows,wrong.run_id))
print(verify_summary(rows,best.run_id))
assert not verify_summary(rows,wrong.run_id)['recommendation_matches_metric']
assert verify_summary(rows,best.run_id)['recommendation_matches_metric']

{'recommended_run_id': '6630883d65784815875f72c0c54aa127', 'best_run_id': '32d4a31f0635446eb714e32f58a7eaca', 'best_rmse_kwh': 1.4342817684933908, 'recommendation_matches_metric': False}
{'recommended_run_id': '32d4a31f0635446eb714e32f58a7eaca', 'best_run_id': '32d4a31f0635446eb714e32f58a7eaca', 'best_rmse_kwh': 1.4342817684933908, 'recommendation_matches_metric': True}


## Prepare a UI comparison
For this UI task, use the local setup: run the printed command in a module-a terminal and open localhost:5050. In hosted Colab, prepare a local matrix instead; a hosted localhost address is not a browser connection. In `linden-demand`, select the five rows, compare parameters and validation metrics, and sort RMSE ascending. Confirm the displayed run ID matches the raw export. API evidence does not substitute for doing this UI task.

In [7]:
print('campus-mlops ui --state',ws.path)
print('Best source run:',best.run_id)
print('Raw CSV:',ws.path/'runs.csv')

campus-mlops ui --state /private/var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-x4mw3x4j/tracking
Best source run: 32d4a31f0635446eb714e32f58a7eaca
Raw CSV: /private/var/folders/tc/l_pmm63j3jn1y1vyr31_4qhc0000gn/T/campus-module-a-x4mw3x4j/tracking/runs.csv


## Recover the record behind the recommendation

Inspect a stored manifest and compare it with the raw run. This connects the selected table row to recoverable configuration and model identity.

In [8]:
recovery=Path(ws.client.download_artifacts(best.run_id,'reproduction/manifest.json'))
saved=json.loads(recovery.read_text())
assert saved['run_id']==best.run_id
assert saved['metrics']['rmse_kwh']==float(best.rmse_kwh)
print(saved['config'],saved['model_uri'])

{'alpha': 0.01, 'autolog': True, 'range_guard': True} models:/m-16e5d3baec5747a782d92fdf449e9fdc


## Your turn: plan, compare and review

Use the matrix plan you wrote before execution. If you choose different values, replace the fixture call with five `ws.run(Config(...))` calls and rank your results. Then: (1) inspect those IDs in the local MLflow UI; (2) reuse the logging-on/off pair to explain which keys autologging adds; (3) obtain an actual approved-assistant response and audit its claims against your table. Reuse these outputs throughout the sprint. The supplied fixture is one valid plan; your chosen values may have a different winner.

## Sprint checkpoint

Harbor needs a justified selection from five controlled runs. Reuse your matrix, UI comparison, logging pair and assistant claim table. Include an actually observed training metric as an excluded observation, keeping it out of validation ranking. Explain comparability, the selected run ID and any unsupported assistant recommendation. Keep the final holdout closed. Complete the Campus client brief in this notebook.

## Finish and keep your work

Save this notebook and the linked project/output files. Keep one sprint submission, reusing the work already completed. Temporary tracking directories disappear when the environment is discarded; save the files before closing it. Continue with the same project in the next sprint.